# `results.html` table to LaTeX

`html/results.html`에 들어 있는 표를 찾아, 논문에 바로 붙여넣을 수 있는 LaTeX 코드로 변환합니다. `TABLE_INDEX`만 바꾸면 원하는 표를 선택할 수 있습니다.

생성된 코드는 `booktabs`, `multirow`, `graphicx` 패키지를 사용하며, 오른쪽에 배치되는 reference-ligand similarity 표에는 `wrapfig`도 필요합니다. HTML의 `rowspan`/`colspan`, 평균 ± 표준편차, 위·아래 화살표, 특수문자, best/second 강조도 함께 변환합니다. 기본 Table 1 출력은 태그를 제외하고 Test와 CL3 filtered 결과만 표시하며, 작은 `± std`는 `\std{...}` 매크로로 출력합니다.

## Configuration

입력 HTML, 출력할 표 번호, 태그 및 크기 옵션을 설정합니다.

In [7]:
from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import re

from bs4 import BeautifulSoup, NavigableString, Tag

# 노트북을 저장소 루트 또는 notebook/ 디렉터리에서 실행해도 동작합니다.
HTML_PATH = Path("html/results.html")
TABLE_INDEX = 1       # 아래 목록에서 원하는 표 번호로 변경
KEEP_BADGES = True    # Table 1 외 표에서 태그를 유지할지 여부
TABLE1_EXCLUDE_TAGS = True
RESIZE_WIDE = True    # 넓은 표에 \resizebox{.98\textwidth}{!}{...} 적용

# Table 1에서는 원본 Test와 최종 CL3 filtered 결과만 출력합니다.
# 각 튜플은 HTML 원본에서 그룹이 시작하는 열과 출력할 컬럼명입니다.
TABLE1_GROUP_HEADERS = [
    (2, "Test (N=1312)"),
    (11, "CL3 filtered (N=733)"),
]
TABLE1_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction} on the original and CL3-filtered "
    r"test sets. \textbf{Bold} marks the best mean and results whose means fall within "
    r"each other's standard-deviation intervals; \underline{underlining} marks the next-best "
    r"mean outside that group."
)

# Similarity performance table: 기존 LP-PDBBind 열은 Table 1과 중복되어 제외합니다.
SIMILARITY_GROUP_HEADERS = [
    (8, "Similarity < 30% (N=453)"),
    (5, "Similarity < 60% (N=813)"),
]
SIMILARITY_EXCLUDE_TAGS = True
REFERENCE_SIMILARITY_EXCLUDE_TAGS = True
SIMILARITY_CAPTION = (
    r"\textbf{LP-PDBBind similarity-stratified binding-affinity prediction} on "
    r"protein-similarity-filtered test sets. \textbf{Bold} marks the best "
    r"mean and results whose means fall within each other's standard-deviation intervals; "
    r"\underline{underlining} marks the next-best mean outside that group."
)


## LaTeX converter

HTML 표 구조와 표별 서식을 LaTeX로 변환하는 함수들입니다.

In [8]:
LINEBREAK = "\ue000"


def resolve_html_path(path: Path) -> Path:
    candidates = [path, Path("notebook") / path]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    tried = ", ".join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(f"results.html을 찾지 못했습니다. 확인한 경로: {tried}")


def escape_latex_text(text: str) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
        "<": r"$<$",
        ">": r"$>$",
    }
    text = "".join(replacements.get(char, char) for char in text)
    unicode_replacements = {
        "\xa0": " ",
        "±": r"$\pm$",
        "ρ": r"$\rho$",
        "σ": r"$\sigma$",
        "Δ": r"$\Delta$",
        "↓": r"$\downarrow$",
        "↑": r"$\uparrow$",
        "→": r"$\rightarrow$",
        "←": r"$\leftarrow$",
        "≤": r"$\leq$",
        "≥": r"$\geq$",
        "≈": r"$\approx$",
        "×": r"$\times$",
        "·": r"$\cdot$",
        "−": "-",
        "—": "---",
        "–": "--",
        "²": r"\textsuperscript{2}",
        "†": r"\textsuperscript{$\dagger$}",
        "\u200a": " ",
    }
    for old, new in unicode_replacements.items():
        text = text.replace(old, new)
    return text


def inline_to_latex(node, keep_badges: bool = True) -> str:
    if isinstance(node, NavigableString):
        return escape_latex_text(str(node))
    if not isinstance(node, Tag) or node.name in {"script", "style"}:
        return ""
    if node.name == "br":
        return LINEBREAK

    content = "".join(inline_to_latex(child, keep_badges) for child in node.children)
    classes = set(node.get("class", []))

    if "tag" in classes:
        if not keep_badges or not content.strip():
            return ""
        return rf"\,\textsuperscript{{\scriptsize {content.strip()}}}"
    if "nsub" in classes:
        return LINEBREAK + content
    if node.name in {"b", "strong"}:
        return rf"\textbf{{{content}}}"
    if node.name in {"i", "em"}:
        return rf"\textit{{{content}}}"
    if node.name == "sub":
        return rf"\textsubscript{{{content}}}"
    if node.name == "sup":
        return rf"\textsuperscript{{{content}}}"
    if "display:block" in node.get("style", "").replace(" ", "").lower():
        return LINEBREAK + content
    return content


def cell_to_latex(
    cell: Tag, keep_badges: bool = True, preserve_emphasis: bool = True
) -> str:
    raw_parts = inline_to_latex(cell, keep_badges).split(LINEBREAK)
    parts = [re.sub(r"\s+", " ", part).strip() for part in raw_parts]
    parts = [part for part in parts if part]
    if not parts:
        return ""

    classes = set(cell.get("class", []))
    style = cell.get("style", "").replace(" ", "").lower()
    is_bold = cell.name == "th" or "best" in classes or bool(re.search(r"font-weight:(?:[7-9]00|bold)", style))
    is_underlined = "second" in classes or "text-decoration:underline" in style
    if preserve_emphasis and is_bold:
        parts = [rf"\textbf{{{part}}}" for part in parts]
    if preserve_emphasis and is_underlined:
        parts = [rf"\underline{{{part}}}" for part in parts]

    if len(parts) == 1:
        return parts[0]
    align = "c" if cell.name == "th" or "metric" in classes else "l"
    return rf"\shortstack[{align}]{{" + r" \\ ".join(parts) + "}"


@dataclass(frozen=True)
class CellPlacement:
    node: Tag
    row: int
    col: int
    rowspan: int
    colspan: int


def build_layout(table: Tag):
    rows = table.find_all("tr")
    occupied = {}
    placements = []
    ncols = 0

    for row_index, row in enumerate(rows):
        col_index = 0
        for cell in row.find_all(["th", "td"], recursive=False):
            while (row_index, col_index) in occupied:
                col_index += 1
            rowspan = max(1, int(cell.get("rowspan", 1)))
            colspan = max(1, int(cell.get("colspan", 1)))
            placement = CellPlacement(cell, row_index, col_index, rowspan, colspan)
            placements.append(placement)
            for rr in range(row_index, row_index + rowspan):
                for cc in range(col_index, col_index + colspan):
                    occupied[(rr, cc)] = placement
            col_index += colspan
        ncols = max(ncols, col_index)
    return rows, occupied, placements, ncols


def table_title(table: Tag, index: int) -> str:
    title_node = table.find_previous("p", class_="table-title")
    if title_node is not None:
        return re.sub(r"\s+", " ", title_node.get_text(" ", strip=True))
    heading = table.find_previous(["h1", "h2", "h3", "h4"])
    return heading.get_text(" ", strip=True) if heading else f"Results table {index + 1}"


def caption_from_title(title: str) -> str:
    caption = re.sub(r"^Table\s+[A-Za-z0-9.]+\s*[·:—-]\s*", "", title, flags=re.IGNORECASE)
    return escape_latex_text(caption)


def label_from_title(title: str, index: int) -> str:
    match = re.match(r"^Table\s+([A-Za-z0-9.]+)", title, flags=re.IGNORECASE)
    stem = match.group(1) if match else str(index + 1)
    stem = re.sub(r"[^a-z0-9]+", "-", stem.lower()).strip("-")
    return f"tab:results-{stem}"


def infer_alignments(placements, ncols: int) -> str:
    alignments = []
    body_cells = [p for p in placements if p.node.name == "td"]
    for col in range(ncols):
        candidates = [p.node for p in body_cells if p.col <= col < p.col + p.colspan]
        cell = candidates[0] if candidates else None
        if cell is None:
            alignments.append("c")
            continue
        classes = set(cell.get("class", []))
        style = cell.get("style", "").replace(" ", "").lower()
        if "text-align:right" in style:
            alignments.append("r")
        elif "metric" in classes or "text-align:center" in style:
            alignments.append("c")
        else:
            alignments.append("l")
    return "".join(alignments)


def parse_mean_std(cell: Tag):
    value_node = cell.find(class_="val")
    if value_node is None:
        return None
    number = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
    value_match = re.search(number, value_node.get_text(strip=True).replace("−", "-"))
    if value_match is None:
        return None
    sd_node = cell.find(class_="sd")
    sd_match = re.search(number, sd_node.get_text(strip=True)) if sd_node else None
    mean = float(value_match.group())
    std = abs(float(sd_match.group())) if sd_match else 0.0
    return mean, std


def apply_std_macro(value: str) -> str:
    """Replace an inline standard deviation with the LaTeX std macro."""
    pattern = r"\s*\$\\pm\$\s*([0-9]+(?:\.[0-9]+)?)"
    return re.sub(pattern, lambda match: rf"\std{{{match.group(1)}}}", value)


def method_name_to_latex(cell: Tag) -> str:
    """Keep only the method name, dropping badges and detail/note spans."""
    parts = []
    for child in cell.children:
        if isinstance(child, NavigableString):
            parts.append(escape_latex_text(str(child)))
        elif isinstance(child, Tag) and child.name in {"b", "strong", "i", "em"}:
            parts.append(inline_to_latex(child, keep_badges=False))
    return re.sub(r"\s+", " ", "".join(parts)).strip()


def format_two_decimal_places(value: str) -> str:
    """Use two decimal places, padding missing precision invisibly."""
    value = value.strip()
    match = re.fullmatch(r"([+-]?\d+)(?:\.(\d+))?", value)
    if match is None:
        return value
    decimals = match.group(2) or ""
    if len(decimals) > 2:
        rounded = Decimal(value).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        return f"{rounded:.2f}"
    if len(decimals) == 2:
        return value
    if len(decimals) == 1:
        return value + r"\phantom{0}"
    return value + r".\phantom{0}\phantom{0}"


def format_pair_two_decimal_places(value: str) -> str:
    return " / ".join(format_two_decimal_places(part) for part in value.split(" / "))


def format_similarity_two_decimal_places(value: str) -> str:
    if " / " in value:
        return format_pair_two_decimal_places(value)
    if value.endswith(r"\%"):
        return format_two_decimal_places(value[:-2]) + r"\%"
    return format_two_decimal_places(value)


def table1_metric_emphasis(rows, placements, metric_columns):
    """Rank cells, requiring mutual mean inclusion for the bold tier."""
    values_by_column = {col: [] for col in metric_columns}
    for placement in placements:
        if placement.col not in values_by_column or placement.node.name != "td":
            continue
        row = rows[placement.row]
        if row.select_one(".tag.leaked, .cat-leaked") is not None:
            continue
        parsed = parse_mean_std(placement.node)
        if parsed is not None:
            values_by_column[placement.col].append((placement.row, *parsed))

    emphasis = {}
    for col, values in values_by_column.items():
        if not values:
            continue
        maximize = (col - 2) % 3 != 2  # Pearson/Spearman ↑, RMSE ↓
        best_mean = (max if maximize else min)(mean for _, mean, _ in values)
        best_std = max(std for _, mean, std in values if abs(mean - best_mean) < 1e-12)
        sota_rows = set()
        for row, mean, std in values:
            mean_gap = abs(mean - best_mean)
            if mean_gap <= best_std and mean_gap <= std:
                emphasis[(row, col)] = "bold"
                sota_rows.add(row)

        remaining = [(row, mean) for row, mean, _ in values if row not in sota_rows]
        if remaining:
            second_mean = (max if maximize else min)(mean for _, mean in remaining)
            for row, mean in remaining:
                if abs(mean - second_mean) < 1e-12:
                    emphasis[(row, col)] = "underline"
    return emphasis


def de_novo_to_latex(table: Tag) -> str:
    """Render the full 260820 meeting-style de novo metric hierarchy."""
    # PosCheck strain energy and steric clashes are excluded here; report their distributions separately.
    # Latest 1.1 values available in notebook/html/260820/260820_meeting.html.
    meeting_rows = {
        "baseline": [
            "0.557", "0.688", "0.741",
            "-5.256 / -6.604", "-6.837 / -7.005", "-7.981 / -8.022",
            "65.3 / 70.0",
        ],
        "decompdiff": [
            "0.485", "0.650", "0.852",
            "-5.278 / -5.507", "-6.168 / -6.129", "-7.258 / -7.449",
            "49.2 / 50.0",
        ],
        "ours": [
            "0.523", "0.677", "0.710",
            "-5.513 / -7.526", "-7.090 / -8.036", "-8.238 / -8.750",
            "67.5 / 74.0",
        ],
    }
    latex_rows = []
    saw_data = False
    for row in table.select("tbody > tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) == 1 and int(cells[0].get("colspan", 1)) > 1:
            if saw_data:
                latex_rows.append(r"\midrule")
            continue
        if len(cells) < 19:
            continue
        saw_data = True
        raw_method = cells[0].get_text(" ", strip=True)
        method = method_name_to_latex(cells[0])
        if raw_method.startswith("DecompDiff") and "ref-informed" in raw_method:
            method = r"DecompDiff\textsubscript{\scriptsize ref-informed}"
        elif raw_method.startswith("DecompDiff") and "ref-free" in raw_method:
            method = r"DecompDiff\textsubscript{\scriptsize ref-free}"
        elif raw_method.startswith("VoxBind"):
            sigma = "0.9" if "0.9" in raw_method else "1.0"
            method = rf"VoxBind\textsubscript{{\scriptsize $\sigma$={sigma}}}"
        elif raw_method.startswith("Ours"):
            method = r"\textbf{VoxBind + CDG}"
        meeting_key = None
        if raw_method.startswith("Ours"):
            meeting_key = "ours"
        elif raw_method.startswith("DecompDiff") and "ref-informed" in raw_method:
            meeting_key = "decompdiff"
        elif raw_method.startswith("VoxBind") and "reproduced" in raw_method and "0.9" in raw_method:
            meeting_key = "baseline"

        if meeting_key is not None:
            values = meeting_rows[meeting_key]
        else:
            sample_quality = [
                cell_to_latex(cells[9], keep_badges=False),   # QED average
                cell_to_latex(cells[11], keep_badges=False),  # SA average
                cell_to_latex(cells[13], keep_badges=False),  # Diversity average
            ]
            vina_pairs = []
            for mean_col, median_col in [(1, 2), (3, 4), (5, 6), (7, 8)]:
                mean = cell_to_latex(cells[mean_col], keep_badges=False)
                median = cell_to_latex(cells[median_col], keep_badges=False)
                vina_pairs.append(f"{mean} / {median}")
            values = [*sample_quality, *vina_pairs]
        values = list(values)
        values[:3] = [format_two_decimal_places(value) for value in values[:3]]
        values[3:] = [format_pair_two_decimal_places(value) for value in values[3:]]
        display_values = [*values[3:], *values[:3]]  # Vina evaluation, then sample quality
        if raw_method.startswith("Ours"):
            latex_rows.append(r"\textbf{VoxBind + C} & \multicolumn{7}{c}{?} \\")
        latex_rows.append(" & ".join([method, *display_values]) + r" \\ ".rstrip())
        if method == "Reference":
            latex_rows.append(r"\midrule")

    caption = (
        r"\textbf{Structure-based drug design} on the CrossDocked2020 benchmark. Vina and "
        r"high-affinity cells report mean / median; arrows indicate the preferred direction."
    )
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        r"    \label{tab:result-de-novo-drug-design}",
        r"    \resizebox{.98\textwidth}{!}{%",
        r"        \begin{tabular}{@{}lccccccc@{}}",
        r"            \toprule",
        r"            \multirow{2}{*}{\textbf{Method}} & \multicolumn{4}{c}{\textbf{Vina evaluation}} & \multicolumn{3}{c}{\textbf{Sample quality}}" + " " + chr(92) * 2,
        r"            \cmidrule(lr){2-5}\cmidrule(lr){6-8}",
        r"            & \textbf{Score $\downarrow$} & \textbf{Min $\downarrow$} & \textbf{Dock $\downarrow$} & \textbf{High aff. $\uparrow$} & \textbf{QED $\uparrow$} & \textbf{SA $\uparrow$} & \textbf{Diversity $\uparrow$}" + " " + chr(92) * 2,
        r"            \midrule",
    ]
    body.extend("            " + line for line in latex_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


def reference_similarity_to_latex(table: Tag) -> str:
    """Transpose reference similarity and render it as a right-side wraptable."""
    methods = []
    values_by_method = []
    for row in table.select("tbody > tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) < 8:
            continue
        method = method_name_to_latex(cells[0])
        bold_match = re.fullmatch(r"\\textbf\{([^{}]+)\}", method)
        methods.append(bold_match.group(1) if bold_match else method)
        values_by_method.append([
            format_similarity_two_decimal_places(cell_to_latex(cell, keep_badges=False))
            for cell in cells[1:8]
        ])

    metric_names = ["Morgan", "Scaffold match", "3D shape", "MACCS", "AtomPair", "RDKit", "Dice"]
    latex_rows = []
    for metric_index, metric in enumerate(metric_names):
        values = [method_values[metric_index] for method_values in values_by_method]
        latex_rows.append(" & ".join([metric, *values]) + r" \\ ".rstrip())

    header = " & ".join([r"\textbf{Metric}", *[rf"\textbf{{{method}}}" for method in methods]]) + " " + chr(92) * 2
    body = [
        r"\begin{wraptable}{r}{0.55\textwidth}",
        r"    \centering",
        r"    \caption{",
        r"        \textbf{Reference-ligand similarity} on the CrossDocked benchmark. Paired similarity values report mean / max; Scaffold match reports the exact-match rate.",
        r"    }",
        r"    \label{tab:results-reference-ligand-similarity}",
        r"    \resizebox{.98\linewidth}{!}{%",
        r"        \begin{tabular}{@{}lccc@{}}",
        r"            \toprule",
        "            " + header,
        r"            \midrule",
    ]
    body.extend("            " + line for line in latex_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{wraptable}",
    ])
    return "\n".join(body)


def table_to_latex(table: Tag, index: int, keep_badges: bool = True, resize_wide: bool = True) -> str:
    if index == 4:
        return de_novo_to_latex(table)
    if index == 5:
        return reference_similarity_to_latex(table)
    rows, occupied, placements, ncols = build_layout(table)
    title = table_title(table, index)
    caption = (
        TABLE1_CAPTION if index == 0
        else SIMILARITY_CAPTION if index == 1
        else caption_from_title(title)
    )
    label = label_from_title(title, index)
    all_alignments = infer_alignments(placements, ncols)
    visible_columns = list(range(ncols))
    custom_headers = {}
    if index == 0:
        custom_headers = dict(TABLE1_GROUP_HEADERS)
        visible_columns = [0, 1] + [
            col
            for start_col, _ in TABLE1_GROUP_HEADERS
            for col in range(start_col, start_col + 3)
        ]
    elif index == 1:
        custom_headers = dict(SIMILARITY_GROUP_HEADERS)
        visible_columns = [0, 1] + [
            col
            for start_col, _ in SIMILARITY_GROUP_HEADERS
            for col in range(start_col, start_col + 3)
        ]
    metric_emphasis = (
        table1_metric_emphasis(rows, placements, set(visible_columns) - {0, 1})
        if index in {0, 1} else {}
    )
    alignments = "".join(all_alignments[col] for col in visible_columns)
    wide = ncols >= 9
    environment = "table*" if wide else "table"

    header_rows = {
        i for i, row in enumerate(rows)
        if row.find_parent("thead") is not None
    }
    if not header_rows and rows and rows[0].find_all("th", recursive=False):
        header_rows = {0}
    last_header_row = max(header_rows) if header_rows else None

    latex_rows = []
    for row_index in range(len(rows)):
        if (
            index in {0, 1}
            and last_header_row is not None
            and row_index > last_header_row + 1
            and rows[row_index].find("td", class_="col-modality", recursive=False)
            is not None
        ):
            latex_rows.append(r"\midrule")
        tokens = []
        visible_index = 0
        while visible_index < len(visible_columns):
            col_index = visible_columns[visible_index]
            placement = occupied.get((row_index, col_index))
            if placement is None:
                tokens.append("")
                visible_index += 1
                continue
            visible_span = [
                col for col in visible_columns
                if placement.col <= col < placement.col + placement.colspan
            ]

            if placement.row == row_index and col_index == visible_span[0]:
                if index in {0, 1} and row_index == 0 and placement.col in custom_headers:
                    header = escape_latex_text(custom_headers[placement.col])
                    value = rf"\textbf{{{header}}}"
                else:
                    if index == 5 and placement.col == 0 and placement.node.name == "td":
                        value = method_name_to_latex(placement.node)
                    else:
                        value = cell_to_latex(
                            placement.node,
                            False
                            if (index == 0 and TABLE1_EXCLUDE_TAGS)
                            or (index == 1 and SIMILARITY_EXCLUDE_TAGS)
                            or (index == 5 and REFERENCE_SIMILARITY_EXCLUDE_TAGS)
                            else keep_badges,
                            preserve_emphasis=(index not in {0, 1} or placement.node.name == "th"),
                        )
                    if index in {0, 1}:
                        value = apply_std_macro(value)
                    emphasis = metric_emphasis.get((row_index, placement.col))
                    if emphasis == "bold":
                        value = rf"\textbf{{{value}}}"
                    elif emphasis == "underline":
                        value = rf"\underline{{{value}}}"
                if placement.rowspan > 1:
                    value = rf"\multirow{{{placement.rowspan}}}{{*}}{{{value}}}"
                visible_colspan = len(visible_span)
                if visible_colspan > 1:
                    value = rf"\multicolumn{{{visible_colspan}}}{{c}}{{{value}}}"
                tokens.append(value)
                visible_index += visible_colspan
            else:
                # 이전 행에서 시작한 multirow가 차지하는 열의 자리표시자입니다.
                span = len(visible_span) if col_index == visible_span[0] else 1
                tokens.append(rf"\multicolumn{{{span}}}{{c}}{{}}" if span > 1 else "")
                visible_index += span

        latex_rows.append(" & ".join(tokens).lstrip() + r" \\ ".rstrip())
        if row_index == last_header_row:
            latex_rows.append(r"\midrule")

    if index in {0, 1}:
        compact_label = (
            "tab:results-binding-affinity"
            if index == 0 else "tab:results-binding-affinity-similarity"
        )
        body = [
            r"\begin{table}[!t]",
            r"    \centering",
            r"    \caption{",
            f"        {caption}",
            r"    }",
            rf"    \label{{{compact_label}}}",
            r"    \resizebox{.98\textwidth}{!}{%",
            rf"        \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"            \toprule",
        ]
        body.extend("            " + line for line in latex_rows)
        body.extend([
            r"            \bottomrule",
            r"        \end{tabular}",
            r"    }",
            r"\end{table}",
        ])
        return "\n".join(body)

    body = [
        r"% Requires: \usepackage{booktabs,multirow,graphicx}",
        rf"\begin{{{environment}}}[t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        rf"    \label{{{label}}}",
    ]
    if wide and resize_wide:
        body.extend([
            r"    \resizebox{.98\textwidth}{!}{%",
            rf"        \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"            \toprule",
        ])
        body.extend("            " + line for line in latex_rows)
        body.extend([
            r"            \bottomrule",
            r"        \end{tabular}",
            r"    }",
        ])
    else:
        body.extend([
            rf"    \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"        \toprule",
        ])
        body.extend("        " + line for line in latex_rows)
        body.extend([
            r"        \bottomrule",
            r"    \end{tabular}",
        ])
    body.append(rf"\end{{{environment}}}")
    return "\n".join(body)


## Results table output

HTML에서 표 목록을 확인하고 `TABLE_INDEX`로 선택한 LaTeX 표를 출력합니다.

In [9]:
resolved_html_path = resolve_html_path(HTML_PATH)
soup = BeautifulSoup(resolved_html_path.read_text(encoding="utf-8"), "html.parser")
tables = soup.find_all("table")

print(f"Source: {resolved_html_path}")
print(f"Found {len(tables)} tables\n")
for index, table in enumerate(tables):
    rows, _, _, ncols = build_layout(table)
    print(f"[{index}] {len(rows)} rows × {ncols} columns | {table_title(table, index)}")


Source: /home/shpark/prj-denovo/VoxBind/notebook/html/results.html
Found 7 tables

[0] 20 rows × 14 columns | Table 1a · Test metrics across cleaning tiers — mean ± std (3 seeds; deterministic zero-shot = 1 pass)
[1] 20 rows × 11 columns | Table 1b · Generalization under protein-sequence novelty (mean ± std; deterministic zero-shot = 1 pass)
[2] 20 rows × 6 columns | Table 2 · Held-out generalization — PDBbind 2019 temporal holdout
[3] 18 rows × 11 columns | Table 3 · CDG-encoder ablation — val ρ / test r / test ρ / RMSE (5 seeds, MSE head)
[4] 17 rows × 19 columns | Table 4 · De novo generation — CrossDocked benchmark
[5] 20 rows × 8 columns | Table A1 · CASF-2016 strict protein-novel cohorts
[6] 5 rows × 6 columns | Table B1 · CASP16 chymase (L1000) — held-out affinity (3-seed mean ± std)


In [10]:
if not 0 <= TABLE_INDEX < len(tables):
    raise IndexError(f"TABLE_INDEX는 0부터 {len(tables) - 1} 사이여야 합니다.")

latex_tables = [
    table_to_latex(table, index, keep_badges=KEEP_BADGES, resize_wide=RESIZE_WIDE)
    for index, table in enumerate(tables)
]
latex = latex_tables[TABLE_INDEX]
print(latex)

# 필요하면 아래 두 줄의 주석을 풀어 .tex 파일로도 저장할 수 있습니다.
# output_path = Path(f"results_table_{TABLE_INDEX + 1}.tex")
# output_path.write_text(latex + "\n", encoding="utf-8")


\begin{table}[!t]
    \centering
    \caption{
        \textbf{LP-PDBBind similarity-stratified binding-affinity prediction} on protein-similarity-filtered test sets. \textbf{Bold} marks the best mean and results whose means fall within each other's standard-deviation intervals; \underline{underlining} marks the next-best mean outside that group.
    }
    \label{tab:results-binding-affinity-similarity}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}llcccccc@{}}
            \toprule
            \multirow{2}{*}{\textbf{Input}} & \multirow{2}{*}{\textbf{Method}} & \multicolumn{3}{c}{\textbf{Similarity $<$ 30\% (N=453)}} & \multicolumn{3}{c}{\textbf{Similarity $<$ 60\% (N=813)}} \\
            &  & \textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$} & \textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$} \\
            \midrule
            \multirow{3}{*}{seq/SMILES} & HonestAffinity & 0.402\std{0.042} & 0.385\st

## Compact right-side ablation table

C/CDG 입력과 ChannelViT channel grouping을 비교하는 오른쪽 `wraptable`을 출력합니다.

In [11]:
def compact_ablation_wraptable(table: Tag) -> str:
    """Render the C/CDG input and channel-group ablation as a right wraptable."""
    rows_by_number = {}
    for row in table.find_all("tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) < 9:
            continue
        number_match = re.search(r"\d+", cells[0].get_text(strip=True))
        if number_match is not None:
            rows_by_number[int(number_match.group())] = cells

    # C, channel-separated CDG, and grouped-channel CDG, in comparison order.
    row_specs = [
        (2, "C", r"$[7,4]$", None),
        (17, "C+D+G", r"$[7,4,1,1]$", "underline"),
        (1, "C+D+G", r"$[7,4,2]$", "bold"),
    ]
    missing = [number for number, *_ in row_specs if number not in rows_by_number]
    if missing:
        raise ValueError(f"Ablation rows not found: {missing}")

    latex_rows = []
    for number, input_name, channels, emphasis in row_specs:
        cells = rows_by_number[number]
        metrics = [cell_to_latex(cells[index], keep_badges=False) for index in (6, 7, 8)]
        if emphasis == "bold":
            metrics = [rf"\textbf{{{value}}}" for value in metrics]
        elif emphasis == "underline":
            metrics = [rf"\underline{{{value}}}" for value in metrics]
        latex_rows.append(" & ".join([input_name, channels, *metrics]) + r" \\ ".rstrip())

    body = [
        r"\begin{wraptable}{r}{0.45\textwidth}",
        r"    \centering",
        r"    \caption{",
        r"        \textbf{Design choice ablation} on the LP-PDBBind test set (N=1320). \textbf{Bold} marks the best result",
        r"    }",
        r"    \label{tab:ablation-cdg-channels}",
        r"    \resizebox{.98\linewidth}{!}{%",
        r"        \begin{tabular}{@{}ll|ccc@{}}",
        r"            \toprule",
        r"            \textbf{Input} & \textbf{Channels} & \textbf{\textit{r} $\uparrow$} & \textbf{$\rho$ $\uparrow$} & \textbf{RMSE $\downarrow$}" + " " + chr(92) * 2,
        r"            \midrule",
    ]
    body.extend("            " + line for line in latex_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{wraptable}",
    ])
    return "\n".join(body)


ablation_wraptable_latex = compact_ablation_wraptable(tables[3])
print(ablation_wraptable_latex)


\begin{wraptable}{r}{0.45\textwidth}
    \centering
    \caption{
        \textbf{Design choice ablation} on the LP-PDBBind test set (N=1320). \textbf{Bold} marks the best result
    }
    \label{tab:ablation-cdg-channels}
    \resizebox{.98\linewidth}{!}{%
        \begin{tabular}{@{}ll|ccc@{}}
            \toprule
            \textbf{Input} & \textbf{Channels} & \textbf{\textit{r} $\uparrow$} & \textbf{$\rho$ $\uparrow$} & \textbf{RMSE $\downarrow$} \\
            \midrule
            C & $[7,4]$ & 0.631 & 0.596 & 1.429 \\
            C+D+G & $[7,4,1,1]$ & \underline{0.650} & \underline{0.634} & \underline{1.422} \\
            C+D+G & $[7,4,2]$ & \textbf{0.664} & \textbf{0.646} & \textbf{1.392} \\
            \bottomrule
        \end{tabular}
    }
\end{wraptable}


## CL1 and CL2 filtered table output

Test/CL3 형식을 재사용해 CL1 및 CL2 filtering 결과를 각각 출력합니다.

In [12]:
def table1_cl1_cl2_latex() -> str:
    """Render CL1 and CL2 filtered results side by side."""
    global TABLE1_GROUP_HEADERS, TABLE1_CAPTION
    original_headers = TABLE1_GROUP_HEADERS
    original_caption = TABLE1_CAPTION
    try:
        TABLE1_GROUP_HEADERS = [
            (5, "CL1 filtered (N=1166)"),
            (8, "CL2 filtered (N=1149)"),
        ]
        TABLE1_CAPTION = original_caption.replace("original and CL3-filtered test sets", "CL1- and CL2-filtered test sets")
        latex = table_to_latex(tables[0], 0, keep_badges=KEEP_BADGES, resize_wide=RESIZE_WIDE)
    finally:
        TABLE1_GROUP_HEADERS = original_headers
        TABLE1_CAPTION = original_caption
    return latex.replace(r"\label{tab:results-binding-affinity}", r"\label{tab:results-binding-affinity-cl1-cl2-filtered}")


cl1_cl2_latex = table1_cl1_cl2_latex()
print(cl1_cl2_latex)


\begin{table}[!t]
    \centering
    \caption{
        \textbf{LP-PDBBind binding-affinity prediction} on the CL1- and CL2-filtered test sets. \textbf{Bold} marks the best mean and results whose means fall within each other's standard-deviation intervals; \underline{underlining} marks the next-best mean outside that group.
    }
    \label{tab:results-binding-affinity-cl1-cl2-filtered}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}llcccccc@{}}
            \toprule
            \multirow{2}{*}{\textbf{Input}} & \multirow{2}{*}{\textbf{Method}} & \multicolumn{3}{c}{\textbf{CL1 filtered (N=1166)}} & \multicolumn{3}{c}{\textbf{CL2 filtered (N=1149)}} \\
            &  & \textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$} & \textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$} \\
            \midrule
            \multirow{3}{*}{seq/SMILES} & HonestAffinity & 0.488\std{0.010} & 0.468\std{0.011} & 1.599\std{0.024} & 